In [1]:
"""Clustering burst features: k-means and HDBSCAN
==============================================

A burst table is a point cloud — one row per burst, one column per feature
(FRET efficiency :math:`E`, stoichiometry :math:`S`, donor lifetime, size, …).
Two questions recur: *how many populations are there* and *which burst belongs
to which*. tttrlib ships the two standard answers as compiled kernels, ported
bit-for-bit from ChiSurf and validated against scikit-learn:

**k-means** (`tttrlib.kmeans`)
    Lloyd's algorithm with greedy k-means++ seeding (2 + ⌊ln k⌋ candidate
    draws per centre, Arthur & Vassilvitskii 2007). The seeding consumes
    *caller-supplied* uniforms, so a fit is reproducible from a stream you own —
    the kernel draws no randomness. From the same seed it lands on the same
    fixed point as scikit-learn's ``KMeans(algorithm="lloyd")`` to 1e-14.

**HDBSCAN** (`tttrlib.hdbscan`, Campello, Moulavi & Sander 2013)
    Density-based, no ``k``. One call runs the whole pipeline —
    ``core_distances`` → ``mutual_reachability_mst`` →
    ``hdbscan_condensed_tree`` → ``hdbscan_select_clusters`` →
    ``hdbscan_label_points`` → ``hdbscan_membership_strengths`` — and those
    kernels stay separate so a caller with its own cluster-selection policy can
    step in between them. Fed either side's tree, tttrlib and scikit-learn's
    ``HDBSCAN`` produce identical partitions and identical membership
    strengths, under excess-of-mass and leaf selection alike.

This example simulates two FRET populations plus noise bursts and runs both.
"""
import numpy as np
import matplotlib.pyplot as plt

import tttrlib

rng = np.random.default_rng(7)

In [2]:
# Simulated burst table
# ---------------------
# Two doubly-labelled populations (low-E and high-E), a donor-only tail, and a
# few noise bursts. Features: E, S, donor lifetime (ns), log burst size.
def population(n, e, s, tau, size):
    return np.column_stack([
        np.clip(rng.normal(e, 0.05, n), 0, 1),
        np.clip(rng.normal(s, 0.04, n), 0, 1),
        rng.normal(tau, 0.25, n),
        np.log10(rng.lognormal(np.log(size), 0.3, n)),
    ])

X = np.vstack([
    population(500, 0.25, 0.55, 3.2, 120),   # low FRET
    population(400, 0.75, 0.55, 1.4, 110),   # high FRET
    population(250, 0.05, 0.92, 3.9, 90),    # donor-only
    np.column_stack([rng.uniform(0, 1, 60), rng.uniform(0, 1, 60),
                     rng.uniform(0.5, 4.5, 60), rng.uniform(1.5, 2.6, 60)]),  # noise
])
truth = np.repeat([0, 1, 2, -1], [500, 400, 250, 60])
X = np.ascontiguousarray(X, dtype=np.float64)

# standardise so no feature dominates the Euclidean distance
Xs = (X - X.mean(0)) / X.std(0)

In [3]:
# k-means with a caller-owned random stream
# ------------------------------------------
# ``n_init`` restarts × ``n_clusters`` centres × ``(2 + int(ln k))`` trials
# uniforms; the best restart (lowest inertia of the *returned* centres) wins.
k, n_init = 3, 5
# (`kmeans_n_uniforms(k, n_init)` is that count; `kmeans_uniforms(k, n_init,
# seed)` draws a stream of the right length from numpy's default_rng -- the
# seed is still yours.)
uniforms = tttrlib.kmeans_uniforms(k, n_init, seed=2026)
centres, labels_km, stats = tttrlib.kmeans(Xs, k, uniforms, n_init, 300, 1e-12)
centres = np.asarray(centres).reshape(k, -1)
labels_km = np.asarray(labels_km)
print(f"k-means: inertia {stats[0]:.1f}, {int(stats[1])} Lloyd sweeps in the winning restart")

k-means: inertia 1454.3, 5 Lloyd sweeps in the winning restart


In [4]:
# HDBSCAN, and what it says about the bursts it is unsure of
# ----------------------------------------------------------
# ``min_cluster_size`` is the smallest population worth calling one, and
# ``min_samples`` how conservative the density estimate is. Nothing else is
# chosen: the number of clusters comes out of the data, and a burst that
# belongs to no population comes back labelled ``-1`` rather than forced into
# the nearest one.
#
# ``probabilities`` is the membership strength — the density at which a burst
# left its cluster over the density at which that cluster died. Thresholding it
# is how one drops the bursts on a population's edge before pooling photons,
# which is usually what a burst-wise classification is *for*.
result = tttrlib.hdbscan(Xs, min_cluster_size=25, min_samples=10)
labels_hd, strength = result.labels, result.probabilities
print(f"HDBSCAN: {labels_hd.max() + 1} clusters, "
      f"{np.sum(labels_hd < 0)} bursts labelled noise, "
      f"{np.sum(strength > 0.9)} core members (strength > 0.9)")

HDBSCAN: 3 clusters, 49 bursts labelled noise, 217 core members (strength > 0.9)


In [5]:
# The other selection policy in the same paper, ``"leaf"``, follows the density
# peaks rather than the mass: where excess of mass keeps a broad population
# whole, leaf selection splits it at its sub-peaks. Neither is more correct —
# on populations as separated as these they agree, and on a table with a
# shoulder they will not.
labels_leaf = tttrlib.hdbscan(Xs, 25, 10, cluster_selection_method="leaf").labels
print(f"leaf selection: {labels_leaf.max() + 1} clusters, "
      f"{np.sum(labels_leaf < 0)} noise")

leaf selection: 3 clusters, 49 noise


In [6]:
# The two partitions against the truth
# ------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(13, 4), sharex=True, sharey=True)
for ax, lab, title in zip(axes, (truth, labels_km, labels_hd),
                          ("truth", "k-means (k = 3)", "HDBSCAN")):
    noise = lab < 0
    ax.scatter(X[~noise, 0], X[~noise, 1], c=lab[~noise], cmap="tab10", s=8, vmin=0, vmax=9)
    ax.scatter(X[noise, 0], X[noise, 1], c="0.6", s=6, marker="x", label="noise")
    ax.set_title(title)
    ax.set_xlabel("FRET efficiency E")
axes[0].set_ylabel("stoichiometry S")
axes[2].legend(loc="lower right")
fig.tight_layout()

In [7]:
# k-means must place every point (the noise gets absorbed by the nearest
# centre); HDBSCAN can say "no cluster", which is what a burst table with
# spurious events needs.
def purity(lab, ref):
    ok = 0
    for c in np.unique(lab[lab >= 0]):
        m = lab == c
        ok += np.bincount(ref[m] + 1).max()
    return ok / np.sum(lab >= 0)

print(f"purity of clustered points: k-means {purity(labels_km, truth):.3f}, "
      f"HDBSCAN {purity(labels_hd, truth):.3f}")
plt.show()

purity of clustered points: k-means 0.950, HDBSCAN 0.990


/var/folders/cl/txwq_hq52rl_t0f9xjk3g5c00000gn/T/ipykernel_72107/3018950528.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
